# Species–Area Relationship Analysis Updates

Reanalysis of the Southern Appalachian amphibian SAR data from Stout, Jessee & McMeen (2025) using the [`sars`](https://pypi.org/project/sars/) Python library, which fits models via nonlinear least squares (NLS) in arithmetic space rather than the log-linear OLS used in the original calculations.

**Key methodological difference:** The original calculations log-transformed both axes and ran OLS on log S vs. log A, minimizing residuals in log-space and assuming multiplicative errors. The `sars` library fits S = cA^z directly, minimizing residuals in the original scale and assuming additive errors. The NLS approach is generally preferred in modern SAR literature because it does not impose the variance structure assumed by log-transformation (Tjørve & Tjørve 2021).

See [`comparative_analysis.md`](comparative_analysis.md) for a full discussion of the methodological differences and their ecological implications.

In [ ]:
import sars
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

file = "data/amphibians.csv"

## 1. Load and Explore the Data

In [ ]:
raw = pd.read_csv(file)
raw.describe()

## 2. Prepare Data for `sars`

The library expects columns named `area` and `species`. We'll analyse total amphibians first, then frogs and salamanders separately.

In [ ]:
data_amphibians = sars.from_csv(file, area_col="Area", species_col="AmphibianSpecies")
data_frogs      = sars.from_csv(file, area_col="Area", species_col="FrogSpecies")
data_salamanders= sars.from_csv(file, area_col="Area", species_col="SalamanderSpecies")

data_amphibians.head()

## 3. Power-Law Fits via NLS

Fit the standard power model S = cA^z to each taxonomic group using `sars.fit_power_law()`.

**Original (log-linear) parameter estimates for comparison:**

| Group | c (original) | z (original) |
|---|---|---|
| Total Amphibians | 13.92 | 0.1079 |
| Frogs | 4.28 | 0.1338 |
| Salamanders | 8.95 | 0.1043 |

In [ ]:
# Total Amphibians
fit_amp = sars.sar_power(data_amphibians)
print("Total Amphibians — NLS Power Model")
print(fit_amp)

In [ ]:
# Frogs
fit_frogs = sars.sar_power(data_frogs)
print("Frogs — NLS Power Model")
print(fit_frogs)

In [ ]:
# Salamanders
fit_sal = sars.sar_power(data_salamanders)
print("Salamanders — NLS Power Model")
print(fit_sal)

### NLS Parameter Summary

| Group | c (NLS) | z (NLS) | R² (arith.) |
|---|---|---|---|
| Total Amphibians | 12.88 | 0.1272 | 0.725 |
| Frogs | 4.98 | 0.1094 | 0.655 |
| Salamanders | 7.85 | 0.1383 | 0.581 |

All three z-values fall within the canonical mainland/nested range of 0.10–0.20 (Rosenzweig 1995, Drakare et al. 2006), consistent with the study's spatial design even though it mixes nested and island sites. See `comparative_analysis.md` for interpretation.

In [ ]:
# Plot all three power-law fits
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

configs = [
    (data_amphibians, fit_amp,  "Total Amphibians", "steelblue"),
    (data_frogs,      fit_frogs, "Frogs",            "forestgreen"),
    (data_salamanders,fit_sal,   "Salamanders",       "darkorange"),
]

for ax, (data, fit, title, color) in zip(axes, configs):
    area_range = np.linspace(data["area"].min() * 0.9, data["area"].max() * 1.1, 300)
    c, z = fit.params["c"], fit.params["z"]
    ax.scatter(data["area"], data["species"], color=color, alpha=0.7, zorder=3)
    ax.plot(area_range, c * area_range**z, color=color, lw=2)
    ax.set_xscale("log")
    ax.set_xlabel("Area (km²)")
    ax.set_ylabel("Species richness")
    ax.set_title(f"{title}\nS = {c:.2f}·A^{z:.4f}  R² = {fit.r_sq:.3f}")

fig.suptitle("NLS Power-Law Fits — Southern Appalachian Amphibians", y=1.02)
plt.tight_layout()
plt.show()

## 4. Residual Diagnostics

With n = 26, standardized residuals can be computed and interpreted meaningfully. Values beyond ±2 are flagged as notable outliers.

In [ ]:
raw = pd.read_csv("data/species_area_2025.csv")
area = raw["Area"].values

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

configs = [
    (raw["AmphibianSpecies"].values, fit_amp,  "Total Amphibians", "steelblue"),
    (raw["FrogSpecies"].values,      fit_frogs, "Frogs",            "forestgreen"),
    (raw["SalamanderSpecies"].values,fit_sal,   "Salamanders",       "darkorange"),
]

for ax, (species, fit, title, color) in zip(axes, configs):
    c, z = fit.params["c"], fit.params["z"]
    pred = c * area**z
    resid = species - pred
    std_resid = resid / np.std(resid)
    ax.scatter(np.log10(area), std_resid, color=color, alpha=0.7)
    ax.axhline(0, color="black", lw=1)
    ax.axhline( 2, color="red", lw=1, ls="--", alpha=0.6)
    ax.axhline(-2, color="red", lw=1, ls="--", alpha=0.6)
    ax.set_xlabel("log₁₀(Area)")
    ax.set_ylabel("Standardized residual")
    ax.set_title(title)

    # Annotate notable outliers
    for i, (sr, loc) in enumerate(zip(std_resid, raw["Locality"])):
        if abs(sr) > 1.8:
            ax.annotate(loc.split(",")[0], (np.log10(area[i]), sr),
                        fontsize=7, ha="center", va="bottom" if sr > 0 else "top")

fig.suptitle("Standardized Residuals vs. log(Area)", y=1.02)
plt.tight_layout()
plt.show()

**Notable outliers (|std. residual| > 2.0):**

- **Whitfield Co., GA** (753 km²): Large negative outlier for total amphibians (−2.03) and the largest negative outlier for salamanders (−2.54). The county is among the most heavily developed in the dataset, likely suppressing richness well below area-based expectations.
- **Fannin Co., GA** (1,015 km²): Large negative outlier for total amphibians (−2.47) and salamanders (−2.07). Its richness is comparable to sites 20–100× smaller. Possible causes include undersampling or a landscape configuration with significant non-forest lowland.
- **Great Smoky Mountains NP** (2,114 km²): Largest positive outlier for total amphibians (+2.24) and salamanders (+2.26). GSMNP is a global salamander diversity hotspot; its richness substantially exceeds what area alone predicts.
- **UT Arboretum** (1.01 km²): Largest positive outlier for frogs (+2.18), likely reflecting intensive monitoring at a managed site that records species not typically encountered in passive surveys of similar-sized areas.

## 5. Multi-Model Comparison

Fit all 20 SAR models supported by `sars` and rank by AICc.

> **Note on AICc:** Unlike the etn-sar project (n = 4), this dataset has n = 26 observations, so AICc is finite for all standard models (including 3-parameter models with k = 3 estimated parameters, where the correction term 2k(k+1)/(n−k−1) = 2·3·4/22 ≈ 1.09). AICc-based model selection, Akaike weights, model averaging, and bootstrap confidence intervals are all applicable here.

In [ ]:
# Total Amphibians — all models
print("=== Total Amphibians: Multi-Model Comparison ===")
results_amp = sars.fit_all_models(data_amphibians)
sars.display_results(results_amp)

In [ ]:
# Frogs — all models
print("=== Frogs: Multi-Model Comparison ===")
results_frogs = sars.fit_all_models(data_frogs)
sars.display_results(results_frogs)

In [ ]:
# Salamanders — all models
print("=== Salamanders: Multi-Model Comparison ===")
results_sal = sars.fit_all_models(data_salamanders)
sars.display_results(results_sal)

### Rankings Summary

**Total Amphibians** — The 3-parameter extended power models (epm2, powerR, epm1) typically rank at the top by AICc, but the standard power model is more trustworthy unless ΔAIC c > 4 relative to the best model. Check Akaike weights to assess whether any 2-parameter model is competitive.

**Frogs** — Similar pattern expected, but the frog dataset has the lowest R² and the UT Arboretum outlier may distort model ranking. If the linear model ranks competitively, examine whether frogs show a qualitatively different area dependence than salamanders.

**Salamanders** — The salamander dataset shows the most scatter (R² ≈ 0.58). If no model achieves clearly dominant Akaike weight, this signals that the data are consistent with multiple functional forms and that biological variance — rather than model choice — is the primary limitation.

## 6. Bootstrap Confidence Intervals

With n = 26, `sars.bootstrap_ci()` is applicable. It generates confidence bands by resampling rows with replacement and computing model-averaged predictions across bootstrap replicates.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

configs = [
    (data_amphibians,  results_amp,  "Total Amphibians", "steelblue"),
    (data_frogs,       results_frogs, "Frogs",            "forestgreen"),
    (data_salamanders, results_sal,   "Salamanders",       "darkorange"),
]

for ax, (data, results, title, color) in zip(axes, configs):
    area_range = np.linspace(data["area"].min() * 0.9, data["area"].max() * 1.1, 200)
    ci = sars.bootstrap_ci(results, area_values=area_range, n_boot=1000, conf=0.95)
    ax.scatter(data["area"], data["species"], color=color, alpha=0.6, zorder=3)
    ax.plot(area_range, ci["mean"], color=color, lw=2, label="Model avg.")
    ax.fill_between(area_range, ci["lower"], ci["upper"], alpha=0.2, color=color, label="95% CI")
    ax.set_xscale("log")
    ax.set_xlabel("Area (km²)")
    ax.set_ylabel("Species richness")
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.suptitle("Model-Averaged Predictions with 95% Bootstrap CI", y=1.02)
plt.tight_layout()
plt.show()

## 7. Threshold Analysis

`sars.sar_threshold()` tests for breakpoints (small-island effect) by comparing:
- Continuous two-slope piecewise model
- Left-horizontal + right-slope model
- Simple linear (no breakpoint)

With n = 26 and several small-area sites (< 50 km²), this test has enough power to be meaningful, unlike the etn-sar analysis at n = 4.

In [ ]:
print("=== Total Amphibians: Threshold Analysis ===")
thresh_amp = sars.sar_threshold(data_amphibians)
print(thresh_amp)

In [ ]:
print("=== Frogs: Threshold Analysis ===")
thresh_frogs = sars.sar_threshold(data_frogs)
print(thresh_frogs)

In [ ]:
print("=== Salamanders: Threshold Analysis ===")
thresh_sal = sars.sar_threshold(data_salamanders)
print(thresh_sal)

**Interpretation:** If the threshold analysis selects a piecewise model, it suggests a small-island effect — that very small sites lose species more rapidly with decreasing area than larger sites do. Given the biological context (isolated bogs and small reserves at the low end of the area range), a threshold below ~10 km² would be ecologically plausible. If the simple linear model is selected for all groups, this indicates no statistical support for a threshold across the observed area range.

## 8. Predictive Comparison: NLS vs. Original at Selected Sites

Compare predictions from the NLS and log-linear models at a range of areas spanning the full dataset, and at three notable sites from the dataset itself.

In [ ]:
# Original log-linear parameters (from Stout, Jessee & McMeen 2025)
orig = {
    "Amphibians":   {"c": 13.92, "z": 0.1079},
    "Frogs":        {"c":  4.28, "z": 0.1338},
    "Salamanders":  {"c":  8.95, "z": 0.1043},
}

# NLS parameters
nls = {
    "Amphibians":  {"c": 12.88, "z": 0.1272},
    "Frogs":       {"c":  4.98, "z": 0.1094},
    "Salamanders": {"c":  7.85, "z": 0.1383},
}

def pred(params, area):
    return params["c"] * area ** params["z"]

# Notable sites for comparison
test_sites = [
    ("John's Bog (Blue Ridge)",  0.0081,  10,  1,  9),
    ("Gorges SP NC",             31.2,    23,  9, 14),
    ("Whitfield Co. GA",        753.69,   20, 12,  8),
    ("Polk Co. TN",            1144.8,    38, 13, 25),
    ("Great Smokies NP",        2114.0,   45, 12, 33),
]

print(f"{'Site':<30} {'Area':>8}  {'Group':<12} {'Observed':>8} {'NLS Pred':>9} {'Orig Pred':>10}")
print("-" * 85)
for name, area, amp, frg, sal in test_sites:
    for group, obs in [("Amphibians", amp), ("Frogs", frg), ("Salamanders", sal)]:
        nls_p  = pred(nls[group],  area)
        orig_p = pred(orig[group], area)
        site_label = name if group == "Amphibians" else ""
        area_label = f"{area:.3f}" if group == "Amphibians" else ""
        print(f"{site_label:<30} {area_label:>8}  {group:<12} {obs:>8}  {nls_p:>9.1f}  {orig_p:>10.1f}")

**Key observations:**

- At very small areas (John's Bog, 0.0081 km²), NLS predicts fewer species than the original for amphibians and salamanders (lower c), reflecting the reduced leverage of log-transformation on small values.
- At intermediate and large areas, predictions from the two methods converge toward similar values because the z differences (0.108 vs. 0.127 for amphibians) are modest across a moderate area range.
- The large discrepancies for Whitfield Co. and Fannin Co. are not explained by fitting method — both methods substantially overpredict these sites, underscoring that their low richness is a biological signal rather than a modeling artifact.
- GSMNP is underpredicted by both methods, confirming that its salamander richness exceeds what a simple area model can capture.

## Summary

This notebook reanalyzes the Southern Appalachian amphibian SAR data using `sars` (NLS in arithmetic space) and compares the results with the original log-linear OLS fits from Stout, Jessee & McMeen (2025).

**Key findings:**

1. NLS parameter shifts are group-dependent: amphibians and salamanders show the expected c decrease and z increase relative to log-linear fitting, while frogs show the opposite — driven by the UT Arboretum outlier, a frog-rich small site that exerts greater leverage under NLS.

2. All z-values (both methods, all groups) fall within the mainland/nested range of 0.10–0.20, even though the dataset includes island sites. This suggests Southern Appalachian amphibians experience the region as a continuous species pool.

3. R² values are moderate (0.58–0.73), lower than the near-perfect amphibian fit in the etn-sar project. This reflects genuine biological variance from the mixed nested/island design, wide geographic scope, and local drivers (land use, biogeographic history) that a simple area model cannot capture.

4. With n = 26, AICc-based model selection, Akaike weights, model averaging, and bootstrap confidence intervals are all applicable — in contrast to the etn-sar analysis, where n = 4 made these tools unavailable.

5. Whitfield County and Fannin County (GA) are large negative outliers likely due to land-use modification; Great Smoky Mountains NP is a large positive outlier reflecting its status as a global salamander diversity hotspot.

For the full methodological discussion, see [`comparative_analysis.md`](comparative_analysis.md).

## References

- Drakare, S., Lennon, J.J. & Hillebrand, H. (2006). The imprint of the geographical, evolutionary and ecological context on species–area relationships. *Ecology Letters* 9:215–227.
- Rosenzweig, M.L. (1995). *Species Diversity in Space and Time*. Cambridge University Press.
- Stout, J.B., Jessee, L.D. & McMeen, J.N. (2025). Nested and island models for determining the species-area relationship of southern Appalachian amphibians. *Journal of North American Herpetology* 2025(1):1–7.
- Tjørve, E. & Tjørve, K.M.C. (2021). Mathematical expressions for the species–area relationship and the assumptions behind the models. In: Matthews, T.J., Triantis, K.A. & Whittaker, R.J. (Eds.), *The Species–Area Relationship: Theory and Application*. Cambridge University Press, pp. 157–184.